## Content-based filtering

In this document, we implement a content-based filtering recommender system that suggests movies by observing its attributes and player's profile.

### Libraries

In [1]:
import numpy as np
import pandas as pd
import sys
import os
from pathlib import Path
import pandas as pd
import ast
import re
import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pickle


### Loading dataset

In [2]:
# Get the current notebook's directory
PROJECT_ROOT = Path(os.getcwd()).parent  # Assuming 'notebooks/' is inside the project root

# Add it to sys.path
sys.path.append(str(PROJECT_ROOT))

# Now you can import your config module
import isa_project_1.config as config

print("opening dataset...")
# here we read the respective output file where the dataset was stored 
movie_df = pd.read_csv(config.INTERIM_DATASETS['tmdb_5000_movies.csv'])
print("success!")

2025-03-02 19:56:52.635 | INFO     | ML_pipeline.isa_project_1.config:<module>:11 - PROJ_ROOT path is: C:\Users\fmojt\Documents\DataAnalysisProjects\Mini-project1_ISA\deployment\ML_pipeline


2025-03-02 19:56:52.655 | INFO     | isa_project_1.config:<module>:11 - PROJ_ROOT path is: C:\Users\fmojt\Documents\DataAnalysisProjects\Mini-project1_ISA\deployment\ML_pipeline
opening dataset...
success!


### Data Processing

First the data must be properly preprocessed. This includes following actions:

1) **Replacing NaNs**
2) **Extracting key data from JSON-like objects**
3) **Concentating the independent variables (predictors)**
4) **Futher text preprocessing (to lowercase, removing non-character letters, removing stopwords)**

#### nltk library

nltk is a useful python library that work with human language data. In the content of our project, it's going to help by **providing stopwords** that are available in English language.

In [3]:
# downloading stopwords (only needed once)
nltk.download('stopwords')
# here we use the unordered set (represented by hash table) to optimize stopword search
stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\fmojt\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


#### Replacing NaN values

In [4]:
# filling NaN values with empty strings
movie_df['overview'] = movie_df['overview'].fillna('')
movie_df['tagline'] = movie_df['tagline'].fillna('')

#### Extracting key attributes from JSON-like strings

Attributes *genres* and *keywords* which make up our predictors are stored in JSON-like lists. We'are going to solve this issue by extracting only the **name key** and replacing the entire list with it.

For this purpose, we have declared a separate function and defined it below.

In [5]:
# Function to extract text from JSON-like columns
def extract_names(text: str, key: str):
    """Converts the provided JSON-like list into a string of one its keys delimited by whitespace.

    Args:
        text (str): the JSON list to be converted
        key (str): the element key to extract

    Returns:
        str: final string of extracted keys
    """
    try:
        # convert the string dictionary into python dictionary
        lst = ast.literal_eval(text)

        # extracting the key
        return ' '.join([i[key] for i in lst])
    except (ValueError, SyntaxError):
        return ''

Now to replace the strings we simply call the function.

In [6]:
# Apply extraction function to genres and keywords
movie_df['genres'] = movie_df['genres'].apply(extract_names, key='name')
movie_df['keywords'] = movie_df['keywords'].apply(extract_names, key='name')

#### Concentatining the independent variables

It's a good practice to combine all text predictors into single text by creating a separate feature *combined_text*. 

In [7]:
# combinining relevant features into single text
movie_df['combined_text'] = movie_df['overview'] + ' ' + movie_df['tagline'] + ' ' + movie_df['genres'] + ' ' + movie_df['keywords']

#### Further text preprocessing

Finally the combined text needs some additional preprocessing:
1) making the text lowercase
2) removing non-word characters (!@#$%^&, etc.)
3) removing stopwords

In [8]:
# Text preprocessing
def preprocess_text(text: str):
    """Converts the text to lowercase, removes all special characters and stopwords.

    Args:
        text (str): the text to be processed

    Returns:
        str: the preprocessed text
    """

    text = text.lower()
    text = re.sub(r'\W+', ' ', text)
    text = ' '.join([word for word in text.split() if word not in stop_words])
    return text

movie_df['processed_text'] = movie_df['combined_text'].apply(preprocess_text)

#### Vectorizing the text using TF-IFD

We use the **TfidfVectorizer** from *sklearn* to vectorize all the movies. By doing so we'll get a TFIDF matrix containing documents (rows) and all words (columns). Every value represent measure of how important a word is to the entire corpus.

In [9]:
# TF-IDF Vectorization
# TF-IDF is a statistical measure that evaluates how important a word (term) is
# to a document or collection (corpus)

# TF-IDF = TF x IDF
# TF = occur_of_term_in_document / total_words_in_document
# IDF = total_num_of_documents / num_of_documents_containing_the_term
tfidf_vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf_vectorizer.fit_transform(movie_df['processed_text'])

## Modelling

After data are successfully preprocessed we can start modelling. First feature that is defined by our business goal is **the user profile**. In our context, the user profile is a complete a history of movies that they liked or somehow interacted with.

For the development purposes, we've decided to use a **list of n randomly selected movies** from dataset. Note that n is represented byh *USER_LIKED_MOVIES* constant.

In [10]:
USER_LIKED_MOVIES = 100

# Randomly select n movies from the entire dataset
user_liked_movies = movie_df.sample(n=USER_LIKED_MOVIES, random_state=42)['title'].tolist()

print("Interacted movies:", user_liked_movies)

Interacted movies: ['I Spy', 'Split Second', 'Gossip', 'Vicky Cristina Barcelona', 'Harry Potter and the Half-Blood Prince', 'AVP: Alien vs. Predator', 'The Contender', 'Meet the Parents', 'Away We Go', 'Sleep Dealer', 'In Her Line of Fire', 'Beyond the Lights', 'The Front Page', 'Taxi Driver', 'Knight and Day', 'End of Days', 'Code 46', 'Double Take', 'Tidal Wave', 'I Heart Huckabees', 'My Own Private Idaho', 'Bandits', 'Crazy in Alabama', '(500) Days of Summer', 'Extremely Loud & Incredibly Close', 'The Artist', 'Stan Helsing', 'The Relic', 'The Midnight Meat Train', 'Ghost Rider: Spirit of Vengeance', 'My Summer of Love', 'Dark Blue', 'Kill Bill: Vol. 1', "Meek's Cutoff", 'The Master', 'Once Upon a Time in the West', 'The Golden Compass', 'The Monuments Men', 'The Egyptian', 'Free Style', 'X-Men: The Last Stand', 'Rocky', 'Pitch Black', 'Iron Man 2', 'Ruby in Paradise', 'The Smurfs', 'Oliver!', 'The Celebration', 'The Last Witch Hunter', 'Prom Night', 'The Dead Girl', 'Saving Silver

In [11]:
"""'I Spy', 'Split Second', 'Gossip', 'Vicky Cristina Barcelona', 'Harry Potter and the Half-Blood Prince', 'AVP: Alien vs. Predator', 'The Contender', 'Meet the Parents', 'Away We Go', 'Sleep Dealer', 'In Her Line of Fire', 'Beyond the Lights', 'The Front Page', 'Taxi Driver', 'Knight and Day', 'End of Days', 'Code 46', 'Double Take', 'Tidal Wave', 'I Heart Huckabees', 'My Own Private Idaho', 'Bandits', 'Crazy in Alabama', '(500) Days of Summer', 'Extremely Loud & Incredibly Close', 'The Artist', 'Stan Helsing', 'The Relic', 'The Midnight Meat Train', 'Ghost Rider: Spirit of Vengeance', 'My Summer of Love', 'Dark Blue', 'Kill Bill: Vol. 1', "Meek's Cutoff", 'The Master', 'Once Upon a Time in the West', 'The Golden Compass', 'The Monuments Men', 'The Egyptian', 'Free Style', 'X-Men: The Last Stand', 'Rocky', 'Pitch Black', 'Iron Man 2', 'Ruby in Paradise', 'The Smurfs', 'Oliver!', 'The Celebration', 'The Last Witch Hunter', 'Prom Night', 'The Dead Girl', 'Saving Silverman', 'Love Me Tender', 'Little Nicky', 'Incident at Loch Ness', 'Autumn in New York', 'Training Day', 'Hocus Pocus', 'Hardflip', 'The Doors', 'Scott Walker: 30 Century Man', 'Why Did I Get Married?', 'To Kill a Mockingbird', 'Bachelorette', 'Mr. Peabody & Sherman', 'Little Shop of Horrors', 'The Mexican', 'Point Blank', 'Jimmy and Judy', 'Becoming Jane', 'Josie and the Pussycats', 'The Jerky Boys', 'Transporter 2', 'Three', 'The Invention of Lying', 'Hum To Mohabbat Karega', 'Standard Operating Procedure', 'The Oxford Murders', 'The Texas Chainsaw Massacre: The Beginning', 'Dolphin Tale 2', 'Bella', 'All Hat', 'Killers', 'The Legend of Bagger Vance', 'Trust', 'Three Kingdoms: Resurrection of the Dragon', 'Smoke Signals', 'The Sweetest Thing', 'Mission to Mars', 'Thomas and the Magic Railroad', 'Duplicity', 'Dragonslayer', "It's Kind of a Funny Story", 'The Dark Knight', 'Wonder Boys', 'Pirates of the Caribbean: The Curse of the Black Pearl', 'Two Brothers', '25th Hour', 'Poultrygeist: Night of the Chicken Dead', 'Red Dawn'""".replace("'", '"')

'"I Spy", "Split Second", "Gossip", "Vicky Cristina Barcelona", "Harry Potter and the Half-Blood Prince", "AVP: Alien vs. Predator", "The Contender", "Meet the Parents", "Away We Go", "Sleep Dealer", "In Her Line of Fire", "Beyond the Lights", "The Front Page", "Taxi Driver", "Knight and Day", "End of Days", "Code 46", "Double Take", "Tidal Wave", "I Heart Huckabees", "My Own Private Idaho", "Bandits", "Crazy in Alabama", "(500) Days of Summer", "Extremely Loud & Incredibly Close", "The Artist", "Stan Helsing", "The Relic", "The Midnight Meat Train", "Ghost Rider: Spirit of Vengeance", "My Summer of Love", "Dark Blue", "Kill Bill: Vol. 1", "Meek"s Cutoff", "The Master", "Once Upon a Time in the West", "The Golden Compass", "The Monuments Men", "The Egyptian", "Free Style", "X-Men: The Last Stand", "Rocky", "Pitch Black", "Iron Man 2", "Ruby in Paradise", "The Smurfs", "Oliver!", "The Celebration", "The Last Witch Hunter", "Prom Night", "The Dead Girl", "Saving Silverman", "Love Me Tend

After creating the sample of user-liked movies we can now extract the subset of TF-IDF matrix that corresponds to it.

In [12]:
user_liked_movies = pd.Series(user_liked_movies)

# Get all movie vectors for movies the user liked
liked_movie_indices = movie_df[movie_df['title'].isin(user_liked_movies)].index
# we get the subset of the entire tfidf matrix based on user liked movies
liked_movie_vectors = tfidf_matrix[liked_movie_indices]

Now we have a sub-matrix which is composed of USER_LIKED_MOVIES rows and M terms. We need, however, only single vector (1xM) that summarizes all the movies into one row by applying a formula that takes into account all the rows. We've decided to use **mean** to achieve this target.

In [13]:
# mean is calculated of all movies user has interacted with
# the result needs to be converted into 2D array with one row (1 x N)
user_profile_vector = np.asarray(liked_movie_vectors.mean(axis=0)).reshape(1, -1)

#### Applying Cosine Similarity

Now that we have one unified vector we can compare it with the original TFIDF matrix and **calculate their Cosine Similarity**. After that we can create a new dataframe containing the similarities and **extract the top n most similar movies**.

In [14]:
user_similarities = cosine_similarity(user_profile_vector, tfidf_matrix).flatten()

RECOMMENDED_MOVIES = 10
movie_scores = pd.DataFrame({'title': movie_df['title'], 'similarity': user_similarities})
recommended_movies = movie_scores.sort_values(by='similarity', ascending=False).head(RECOMMENDED_MOVIES)

# recommendations = recommend_movies(**user_preferences)
print("Top Recommended Movies:")
# print(recommendations)

# recommended_movies.head(n)
recommended_movies_list = list(recommended_movies['title'])
print(recommended_movies_list)


Top Recommended Movies:
['Bandits', 'The Dead Girl', 'The Last Song', 'Autumn in New York', 'Dead Man Down', 'End of Days', 'Mr. Peabody & Sherman', 'Training Day', "Brooklyn's Finest", 'Code 46']


## Testing dataset

Now in order to test the dataset we're going to use various metrics, like **Precision**, **Recall** or **F1**. In the context of our model, the metrics mean following:

1) Precision - how many movies **were recommended and actually liked** and **were not recommended and not liked** ?
2) Recall - how many movies **were recommended and actually liked** and how many **not liked movies were actually not recommended**?
3) f1 - harmonic mean between Precision and Recall 

In [15]:
# Convert to sets for easy comparison
recommended_set = set(recommended_movies_list)
actual_set = set(user_liked_movies)

# Calculate True Positives, False Positives, and False Negatives
TP = len(recommended_set & actual_set)  # Intersection: movies both recommended and liked
FP = len(recommended_set - actual_set)  # Recommended but not liked
FN = len(actual_set - recommended_set)  # Liked but not recommended

# Compute precision, recall, and F1-score
precision = TP / (TP + FP) if (TP + FP) > 0 else 0
recall = TP / (TP + FN) if (TP + FN) > 0 else 0
f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

# Print results
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-score: {f1:.4f}")

Precision: 0.7000
Recall: 0.0700
F1-score: 0.1273


## Saving TF-IDF matrix&vectorizer

Finally, we save the both the TF-IDF matrix and vectorizer to .pkl files (binary format).

In [16]:
# Save the TF-IDF matrix and vectorizer
with open(config.MODELS_DIR / "tfidf_matrix.pkl", "wb") as f:
    pickle.dump(tfidf_matrix, f)

with open(config.MODELS_DIR / "tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(tfidf_vectorizer, f)

movie_df.to_pickle(config.PROCESSED_DATA_DIR / "movie_df.pkl")


## Conclusion

In this document we have successfully preprocessed predictors and the target variable, created user profile of n randomly-selected interacted movies, calculated the similarity between user profile and other movies, recommended top k most similar movies and, lastly, tested the predictions using metrics precision and recall.

When using the model the number of user-liked movies must be greater or equal to the size of recommend movie list. In the testing we selected a 100 of movies the user has interacted with and then recomended top 10 movies. The metrics that is important to our business goal is the precision because it is important to recommend mainly the movies user liked but it is always good not to recommend those they didn't like. The recall metrics was low because model had tendention to recommend movies user didnt like yet (recommendation system). 